# ADS v2: run

Runs the ads build notebooks in dependency order: 10_Common, then the event family notebooks 11 to 18 in parallel, then 19_EventStream, then 20_ScvAttach, then 21_EpisodeNumbering and 22_Parity; 23_CampaignHistory runs once 10_Common is done. Leave `batch_id` blank for a full run. To **resume** a failed run, set `batch_id` to that run's batch id: notebooks that already succeeded in it (ads_v2.AdsRunLog) are skipped.

To run **some notebooks only**, list them in `only` (for example `19_EventStream, 20_ScvAttach`). Their dependencies outside the list are not run and are assumed to be built already.

The build window (start_time, end_time, effective_date) comes from ADS_v2_00_Setup, which every notebook runs first.


In [ ]:
%%pyspark
# Parameters
batch_id = ""          # blank = new full run; an earlier batch id = resume it
only     = ""          # blank = every notebook; else a comma-separated list, e.g. "19_EventStream, 20_ScvAttach"


In [ ]:
%%pyspark
import datetime, json
from pyspark.sql import functions as F
NOTEBOOKS = json.loads(r"""[{"name":"10_Common","depends":[]},{"name":"11_Motor_Acquisition","depends":["10_Common"]},{"name":"12_Motor_Renewal","depends":["10_Common"]},{"name":"13_Home_Acquisition","depends":["10_Common"]},{"name":"14_Home_Renewal","depends":["10_Common"]},{"name":"15_Motor_InLife","depends":["10_Common"]},{"name":"16_Home_InLife","depends":["10_Common"]},{"name":"17_Van","depends":["10_Common"]},{"name":"18_Travel","depends":[]},{"name":"19_EventStream","depends":["11_Motor_Acquisition","12_Motor_Renewal","13_Home_Acquisition","14_Home_Renewal","15_Motor_InLife","16_Home_InLife","17_Van","18_Travel"]},{"name":"20_ScvAttach","depends":["10_Common","19_EventStream"]},{"name":"21_EpisodeNumbering","depends":["20_ScvAttach"]},{"name":"22_Parity","depends":["20_ScvAttach"]},{"name":"23_CampaignHistory","depends":["10_Common"]}]""")
target_schema = "ads_v2"
import time
batch_id = batch_id or datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d%H%M%S")
if only.strip():
    wanted = [x.strip() for x in only.split(",") if x.strip()]
    unknown = [x for x in wanted if x not in [n["name"] for n in NOTEBOOKS]]
    if unknown:
        raise ValueError(f"unknown notebook(s) in only: {unknown}; choose from {[n['name'] for n in NOTEBOOKS]}")
    NOTEBOOKS = [n for n in NOTEBOOKS if n["name"] in wanted]
selected = {n["name"] for n in NOTEBOOKS}
# Fabric platform faults (e.g. a 504 while runMultiple fetches the notebooks) are retried; failures inside our
# notebooks are not. Each attempt skips notebooks that already succeeded in this batch, so nothing is redone.
# If runMultiple still fails on a platform fault, the notebooks run one at a time (notebookutils.notebook.run), in
# dependency order: slower, but no bulk notebook fetch.
TRANSIENT = ("Fetch notebook content", "GatewayTimeout", "status code: 504", "timed out", "TooManyRequests", "status code: 429")


def succeeded():
    if not spark.catalog.tableExists(f"{target_schema}.AdsRunLog"):
        return set()
    return {r.Notebook for r in spark.table(f"{target_schema}.AdsRunLog")
                .where((F.col("AdsBatchId") == batch_id) & (F.col("Status") == "succeeded")).select("Notebook").distinct().collect()}


finished = False
for attempt in range(1, 3):
    done = succeeded()
    todo = [n for n in NOTEBOOKS if n["name"] not in done]
    print(f"batch {batch_id}, attempt {attempt}: {'already done: ' + ', '.join(sorted(done)) if done else 'nothing done yet'}")
    if not todo:
        finished = True
        break
    print("to run:", ", ".join(n["name"] for n in todo))
    dag = {"activities": [{"name": n["name"], "path": "ADS_v2_" + n["name"], "timeoutPerCellInSeconds": 7200,
                            "args": {"batch_id": batch_id},
                            "dependencies": [d for d in n["depends"] if d not in done and d in selected]} for n in todo],
           "concurrency": 4}
    try:
        notebookutils.notebook.runMultiple(dag)
        finished = True
        break
    except Exception as e:  # noqa: BLE001
        msg = str(e)
        if not any(t in msg for t in TRANSIENT):
            raise
        print(f"platform fault on attempt {attempt}: {msg[:300]}")
        if attempt == 1:
            time.sleep(60)
if not finished:
    print("runMultiple kept failing on platform faults: running the notebooks one at a time")
    for n in NOTEBOOKS:
        if n["name"] in succeeded():
            continue
        print("running", n["name"])
        notebookutils.notebook.run("ADS_v2_" + n["name"], 14400, {"batch_id": batch_id})


In [ ]:
%%pyspark
# Each notebook's start / success / failure in this batch
display(spark.table(f"{target_schema}.AdsRunLog").where(F.col("AdsBatchId") == batch_id).orderBy("LoggedAt"))


In [ ]:
%%pyspark
# The checks each notebook ran on the tables it built (unique keys, no personal data)
display(spark.table(f"{target_schema}.AdsBuildLog").where(F.col("AdsBatchId") == batch_id).orderBy("TableName"))


In [ ]:
%%pyspark
# Event families that need an explanation in writing: no events, events outside the window, or identical to another family
if spark.catalog.tableExists(f"{target_schema}.AdsFamilyCheck"):
    display(spark.table(f"{target_schema}.AdsFamilyCheck")
                 .where((F.col("AdsBatchId") == batch_id) & (F.coalesce(F.col("Flags"), F.lit("")) != ""))
                 .orderBy("EventTypeId"))
